In [1]:
import pandas as pd

## Goal
The goal of this notebook is to set up a features set for the intensity basic classification model.

### Mass
- **f1** = `{MW}` (Molecular Weight)

### Lipophilicity
- **f2** = `{MLOGP}`
- **f3** = `{ALOGP, ALOGP2, MLOGP, MLOGP2}`

### Volume (van der Waals volume)
- **f4** = `VvdwMG`
- **f5** = `VvdwZAZ`
- **f6** = `Sv` (sum of atomic volumes)

### Surface area and shape
- **f7** = `{Morxxu, Morxxm, Morxxv, ...}` - Derived from electron diffraction; encodes 3D atom distribution and relates to molecular shape and boiling point.
- **f8** = `{RDFxxxx...}` - Probability of finding atoms at specific distances; captures molecular size, surface spread, and compactness.
- **f9** = `{L1, L2u, L3u, Tu, Au, ...}` - WHIM descriptors based on 3D coordinates; encode molecular shape, symmetry, and surface anisotropy.
- **f10** = `{HATS*, R*}` - Combines geometry and topology; encodes how atoms are distributed over molecular surface.
- **f11** = `{GATS*}` - Spatial autocorrelation descriptors; captures how properties vary across molecular surface.

### Polarity
- **f12** = `{PDI}` - Higher values indicate stronger interactions and often lower vapor pressure.

### Surface-related descriptors for vapor pressure
- **f13** = `{P_VSA_LogP_2, P_VSA_LogP_3, P_VSA_LogP_4, P_VSA_LogP_5, P_VSA_LogP_7}` - Partitioned van der Waals surface area; captures surface area and polarity distribution.

### Shape/size distribution related to vapor pressure
- **f14** = `{SpDiam_A, SpDiam_D, SpDiam_L, SpDiam_X}`


In [2]:
data_df = pd.read_csv('data/waka_dragon_merged.csv')

In [3]:
# Count how many descriptors from each defined feature group exist in data_df
columns = list(data_df.columns)
column_lookup = {c.lower(): c for c in columns}

feature_groups = {
    "f1_mass": {
        "exact": ["MW"],
        "prefix": [],
    },
    "f2_lipophilicity": {
        "exact": ["MLOGP"],
        "prefix": [],
    },
    "f3_lipophilicity_extended": {
        "exact": ["ALOGP", "ALOGP2", "MLOGP", "MLOGP2"],
        "prefix": [],
    },
    "f4_volume": {
        "exact": ["VvdwMG"],
        "prefix": [],
    },
    "f5_volume": {
        "exact": ["VvdwZAZ"],
        "prefix": [],
    },
    "f6_volume": {
        "exact": ["Sv"],
        "prefix": [],
    },
    "f7_surface_shape_Mor": {
        "exact": ["Morxxu", "Morxxm", "Morxxv"],
        "prefix": ["Mor"],
    },
    "f8_surface_shape_RDF": {
        "exact": ["RDFxxxx"],
        "prefix": ["RDF"],
    },
    "f9_whim": {
        "exact": ["L1", "L2u", "L3u", "Tu", "Au"],
        "prefix": [],
    },
    "f10_geometry_topology": {
        "exact": [],
        "prefix": ["HATS", "R"],
    },
    "f11_spatial_autocorrelation": {
        "exact": [],
        "prefix": ["GATS"],
    },
    "f12_polarity": {
        "exact": ["PDI"],
        "prefix": [],
    },
    "f13_p_vsa_logp": {
        "exact": [
            "P_VSA_LogP_2",
            "P_VSA_LogP_3",
            "P_VSA_LogP_4",
            "P_VSA_LogP_5",
            "P_VSA_LogP_7",
        ],
        "prefix": [],
    },
    "f14_spdiam": {
        "exact": ["SpDiam_A", "SpDiam_D", "SpDiam_L", "SpDiam_X"],
        "prefix": [],
    },
}


def match_exact(names):
    return [column_lookup[n.lower()] for n in names if n.lower() in column_lookup]


def match_prefix(prefixes):
    if not prefixes:
        return []
    matches = []
    for c in columns:
        c_lower = c.lower()
        if any(c_lower.startswith(p.lower()) for p in prefixes):
            matches.append(c)
    return matches


rows = []
for group_name, cfg in feature_groups.items():
    exact_names = cfg["exact"]
    prefixes = cfg["prefix"]

    exact_found = match_exact(exact_names)
    prefix_found = match_prefix(prefixes)

    found_all = sorted(set(exact_found + prefix_found))
    missing_exact = [n for n in exact_names if n.lower() not in column_lookup]

    rows.append(
        {
            "group": group_name,
            "defined_exact": len(exact_names),
            "found_exact": len(exact_found),
            "found_by_prefix": len(prefix_found),
            "total_found": len(found_all),
            "missing_exact": missing_exact,
            "example_found": found_all[:8],
        }
    )

# Keep output in the same order as groups are defined above (f1 -> f14)
summary_df = pd.DataFrame(rows).reset_index(drop=True)
summary_df

,group,defined_exact,found_exact,found_by_prefix,total_found,missing_exact,example_found
0,f10_geometry_topology,0,0,390,390,[],"[HATS0e, HATS0i, HATS0m, HATS0p, HATS0s, HATS0..."
1,f11_spatial_autocorrelation,0,0,48,48,[],"[GATS1e, GATS1i, GATS1m, GATS1p, GATS1s, GATS1..."
2,f12_polarity,1,1,0,1,[],[PDI]
3,f13_p_vsa_logp,5,5,0,5,[],"[P_VSA_LogP_2, P_VSA_LogP_3, P_VSA_LogP_4, P_V..."
4,f14_spdiam,4,4,0,4,[],"[SpDiam_A, SpDiam_D, SpDiam_L, SpDiam_X]"
5,f1_mass,1,1,0,1,[],[MW]
6,f2_lipophilicity,1,1,0,1,[],[MLOGP]
7,f3_lipophilicity_extended,4,4,0,4,[],"[ALOGP, ALOGP2, MLOGP, MLOGP2]"
8,f4_volume,1,1,0,1,[],[VvdwMG]
9,f5_volume,1,1,0,1,[],[VvdwZAZ]
